# 08 — Weighted Whole-Body Control

## 目的
NMPCの現在目標から、全身運動方程式・接触・摩擦・トルク上限を満たす
関節トルクを解く瞬間QPを理解する。

実装対応:
- [`WbcBase.cpp`](https://github.com/qiayuanliao/legged_control/tree/a7f381c0367e98e31c01336e678eef47e304d40d/legged_wbc/src/WbcBase.cpp)
- [`WeightedWbc.cpp`](https://github.com/qiayuanliao/legged_control/tree/a7f381c0367e98e31c01336e678eef47e304d40d/legged_wbc/src/WeightedWbc.cpp)
- `HierarchicalWbc.cpp`（実装あり、既定controllerには未配線）


## 検証範囲に関する必須注記

このprojectでは **ROS2 portを作成・compile・実行していない**。したがってROS2 parityは
**NOT VERIFIED / FAIL-CLOSED** である。上流commit `a7f381c0367e98e31c01336e678eef47e304d40d` はROS1実装であり、
project所有MuJoCo adapterはOCS2のhorizon SQPを瞬時force plannerへ、
Pinocchio/qpOASES WBCをMuJoCo acceleration inverse dynamicsへ置換し、
元のestimator/hardware経路も持たない。保存済み30 scenario dataが示すのはadapter挙動だけで、
上流 `legged_control` の性能でもROS2移行の検証でもない。


In [1]:
# 背景: NMPC出力からmotor torqueへ全身力学を介して変換する。目的: Weighted WBCの42次元QP契約を検査するため、`from pathlib import Path` の依存を明示して再現可能な実行環境を作る。
from pathlib import Path
# 背景: NMPC出力からmotor torqueへ全身力学を介して変換する。目的: Weighted WBCの42次元QP契約を検査するため、`import numpy as np` の依存を明示して再現可能な実行環境を作る。
import numpy as np
# 背景: NMPC出力からmotor torqueへ全身力学を介して変換する。目的: Weighted WBCの42次元QP契約を検査するため、`import matplotlib.pyplot as plt` の依存を明示して再現可能な実行環境を作る。
import matplotlib.pyplot as plt

# 背景: NMPC出力からmotor torqueへ全身力学を介して変換する。目的: Weighted WBCの42次元QP契約を検査するため、`ROOT` を後続計算で使う明示的な中間量として設定する。 数式: `ROOT = Path.cwd()` の演算・変換をPythonで評価する。
ROOT = Path.cwd()
# 背景: NMPC出力からmotor torqueへ全身力学を介して変換する。目的: Weighted WBCの42次元QP契約を検査するため、`for candidate in [ROOT, *ROOT.parents]:` の反復範囲を固定して各sampleを処理する。 数式: `for candidate in [ROOT, *ROOT.parents]:` の演算・変換をPythonで評価する。
for candidate in [ROOT, *ROOT.parents]:
    # 背景: NMPC出力からmotor torqueへ全身力学を介して変換する。目的: Weighted WBCの42次元QP契約を検査するため、`if (candidate / "pyproject.toml").exists():` の条件で安全側の実行分岐を選ぶ。 数式: `if (candidate / "pyproject.toml").exists():` の演算・変換をPythonで評価する。
    if (candidate / "pyproject.toml").exists():
        # 背景: NMPC出力からmotor torqueへ全身力学を介して変換する。目的: Weighted WBCの42次元QP契約を検査するため、`ROOT` を後続計算で使う明示的な中間量として設定する。 数式: `ROOT = candidate` の演算・変換をPythonで評価する。
        ROOT = candidate
        # 背景: NMPC出力からmotor torqueへ全身力学を介して変換する。目的: Weighted WBCの42次元QP契約を検査するため、`break` をこの章の処理順に沿って実行する。
        break

# 背景: NMPC出力からmotor torqueへ全身力学を介して変換する。目的: Weighted WBCの42次元QP契約を検査するため、`np.set_printoptions(precision` を後続計算で使う明示的な中間量として設定する。 数式: `np.set_printoptions(precision=4, suppress=True)` の演算・変換をPythonで評価する。
np.set_printoptions(precision=4, suppress=True)
# 背景: NMPC出力からmotor torqueへ全身力学を介して変換する。目的: Weighted WBCの42次元QP契約を検査するため、直前の式・構造へ `plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True})` の要素または終端を対応付ける。
plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True})
# 背景: NMPC出力からmotor torqueへ全身力学を介して変換する。目的: Weighted WBCの42次元QP契約を検査するため、`print("repository:", ROOT)` の観測値を表示して判定根拠を残す。
print("repository:", ROOT)


repository: /home/takuya/work/mpc_dog


決定変数:
\[
z=[\ddot q(18),F_c(12),\tau(12)]\in\mathbb R^{42}.
\]
硬い運動方程式:
\[
[M,-J^\top,-S^\top]z=-nle.
\]
さらに torque box、立脚足加速度0、遊脚力0、摩擦pyramidをhard constraintにする。

soft taskは遊脚加速度、base加速度、NMPC接触力追従。
A1既定重みは swing 100、base accel 1、contact force 0.01。
したがってNMPCのGRFは命令ではなく、WBCがずらせる弱い目標である。


In [2]:
# 小さな等式制約付きweighted least squares。
# z=[base acceleration a, contact force F, actuator torque tau]
# hard: m*a - F - tau = -m*g
# 背景: NMPC出力からmotor torqueへ全身力学を介して変換する。目的: Weighted WBCの42次元QP契約を検査するため、`solve_toy_wbc` の責務を独立関数として定義する。 数式: `def solve_toy_wbc(w_a=1.0, w_f=0.01, a_des=0.0, f_des=100.0,` の演算・変換をPythonで評価する。
def solve_toy_wbc(w_a=1.0, w_f=0.01, a_des=0.0, f_des=100.0,
                  # 背景: NMPC出力からmotor torqueへ全身力学を介して変換する。目的: Weighted WBCの42次元QP契約を検査するため、`mass` を後続計算で使う明示的な中間量として設定する。 数式: `mass=12.5, g=9.81):` の演算・変換をPythonで評価する。
                  mass=12.5, g=9.81):
    # 背景: NMPC出力からmotor torqueへ全身力学を介して変換する。目的: Weighted WBCの42次元QP契約を検査するため、`Aeq` を後続計算で使う明示的な中間量として設定する。 数式: `Aeq = np.array([[mass, -1.0, -1.0]])` の演算・変換をPythonで評価する。
    Aeq = np.array([[mass, -1.0, -1.0]])
    # 背景: NMPC出力からmotor torqueへ全身力学を介して変換する。目的: Weighted WBCの42次元QP契約を検査するため、`beq` を後続計算で使う明示的な中間量として設定する。 数式: `beq = np.array([-mass*g])` の演算・変換をPythonで評価する。
    beq = np.array([-mass*g])
    # 背景: NMPC出力からmotor torqueへ全身力学を介して変換する。目的: Weighted WBCの42次元QP契約を検査するため、`C` を後続計算で使う明示的な中間量として設定する。 数式: `C = np.array([[w_a, 0, 0], [0, w_f, 0]], float)` の演算・変換をPythonで評価する。
    C = np.array([[w_a, 0, 0], [0, w_f, 0]], float)
    # 背景: NMPC出力からmotor torqueへ全身力学を介して変換する。目的: Weighted WBCの42次元QP契約を検査するため、`d` を後続計算で使う明示的な中間量として設定する。 数式: `d = np.array([w_a*a_des, w_f*f_des])` の演算・変換をPythonで評価する。
    d = np.array([w_a*a_des, w_f*f_des])
    # KKT: [C'C A'; A 0] [z,lambda] = [C'd,b]
    # 背景: NMPC出力からmotor torqueへ全身力学を介して変換する。目的: Weighted WBCの42次元QP契約を検査するため、`H` を後続計算で使う明示的な中間量として設定する。 数式: `H = C.T@C + 1e-9*np.eye(3)` の演算・変換をPythonで評価する。
    H = C.T@C + 1e-9*np.eye(3)
    # 背景: NMPC出力からmotor torqueへ全身力学を介して変換する。目的: Weighted WBCの42次元QP契約を検査するため、`KKT` を後続計算で使う明示的な中間量として設定する。 数式: `KKT = np.block([[H, Aeq.T], [Aeq, np.zeros((1,1))]])` の演算・変換をPythonで評価する。
    KKT = np.block([[H, Aeq.T], [Aeq, np.zeros((1,1))]])
    # 背景: NMPC出力からmotor torqueへ全身力学を介して変換する。目的: Weighted WBCの42次元QP契約を検査するため、`rhs` を後続計算で使う明示的な中間量として設定する。 数式: `rhs = np.r_[C.T@d, beq]` の演算・変換をPythonで評価する。
    rhs = np.r_[C.T@d, beq]
    # 背景: NMPC出力からmotor torqueへ全身力学を介して変換する。目的: Weighted WBCの42次元QP契約を検査するため、`return np.linalg.solve(KKT, rhs)[:3]` の値を次の制御境界へ返す。 数式: `return np.linalg.solve(KKT, rhs)[:3]` の演算・変換をPythonで評価する。
    return np.linalg.solve(KKT, rhs)[:3]

# 背景: NMPC出力からmotor torqueへ全身力学を介して変換する。目的: Weighted WBCの42次元QP契約を検査するため、`for wf in [0.001, 0.01, 0.1, 1.0]:` の反復範囲を固定して各sampleを処理する。
for wf in [0.001, 0.01, 0.1, 1.0]:
    # 背景: NMPC出力からmotor torqueへ全身力学を介して変換する。目的: Weighted WBCの42次元QP契約を検査するため、`z` を後続計算で使う明示的な中間量として設定する。 数式: `z = solve_toy_wbc(w_f=wf)` の演算・変換をPythonで評価する。
    z = solve_toy_wbc(w_f=wf)
    # 背景: NMPC出力からmotor torqueへ全身力学を介して変換する。目的: Weighted WBCの42次元QP契約を検査するため、`residual` を後続計算で使う明示的な中間量として設定する。 数式: `residual = 12.5*z[0] - z[1] - z[2] + 12.5*9.81` の演算・変換をPythonで評価する。
    residual = 12.5*z[0] - z[1] - z[2] + 12.5*9.81
    # 背景: NMPC出力からmotor torqueへ全身力学を介して変換する。目的: Weighted WBCの42次元QP契約を検査するため、`print(f"w_force={wf:5.3f} -> [a,F,tau]={z}, EoM residual={residual:.2e}"…` の観測値を表示して判定根拠を残す。 数式: `print(f"w_force={wf:5.3f} -> [a,F,tau]={z}, EoM residual={residual:.2e}"…` の演算・変換をPythonで評価する。
    print(f"w_force={wf:5.3f} -> [a,F,tau]={z}, EoM residual={residual:.2e}")


w_force=0.001 -> [a,F,tau]=[-0.     99.9228 22.7022], EoM residual=0.00e+00
w_force=0.010 -> [a,F,tau]=[-0.     99.9992 22.6258], EoM residual=0.00e+00
w_force=0.100 -> [a,F,tau]=[ -0.    100.     22.625], EoM residual=0.00e+00
w_force=1.000 -> [a,F,tau]=[ -0.    100.     22.625], EoM residual=0.00e+00


In [3]:
# 実装の行列shapeを組み立てて、転置・符号の契約を確認する。
# 背景: NMPC出力からmotor torqueへ全身力学を介して変換する。目的: Weighted WBCの42次元QP契約を検査するため、`nq, nf, ntau` を後続計算で使う明示的な中間量として設定する。 数式: `nq, nf, ntau = 18, 12, 12` の演算・変換をPythonで評価する。
nq, nf, ntau = 18, 12, 12
# 背景: NMPC出力からmotor torqueへ全身力学を介して変換する。目的: Weighted WBCの42次元QP契約を検査するため、`M` を後続計算で使う明示的な中間量として設定する。 数式: `M = np.eye(nq)` の演算・変換をPythonで評価する。
M = np.eye(nq)
# 背景: NMPC出力からmotor torqueへ全身力学を介して変換する。目的: Weighted WBCの42次元QP契約を検査するため、`J` を後続計算で使う明示的な中間量として設定する。 数式: `J = np.zeros((nf, nq))` の演算・変換をPythonで評価する。
J = np.zeros((nf, nq))
# 背景: NMPC出力からmotor torqueへ全身力学を介して変換する。目的: Weighted WBCの42次元QP契約を検査するため、`S` を後続計算で使う明示的な中間量として設定する。 数式: `S = np.zeros((ntau, nq)); S[:, 6:] = np.eye(ntau)` の演算・変換をPythonで評価する。
S = np.zeros((ntau, nq)); S[:, 6:] = np.eye(ntau)
# 背景: NMPC出力からmotor torqueへ全身力学を介して変換する。目的: Weighted WBCの42次元QP契約を検査するため、`A_eom` を後続計算で使う明示的な中間量として設定する。 数式: `A_eom = np.c_[M, -J.T, -S.T]` の演算・変換をPythonで評価する。
A_eom = np.c_[M, -J.T, -S.T]
# 背景: NMPC出力からmotor torqueへ全身力学を介して変換する。目的: Weighted WBCの42次元QP契約を検査するため、`assert A_eom.shape == (18, 42)` を不変条件として即時検査する。 数式: `assert A_eom.shape == (18, 42)` の演算・変換をPythonで評価する。
assert A_eom.shape == (18, 42)
# 背景: NMPC出力からmotor torqueへ全身力学を介して変換する。目的: Weighted WBCの42次元QP契約を検査するため、`print("EoM matrix shape:", A_eom.shape)` の観測値を表示して判定根拠を残す。
print("EoM matrix shape:", A_eom.shape)
# 背景: NMPC出力からmotor torqueへ全身力学を介して変換する。目的: Weighted WBCの42次元QP契約を検査するため、`print("decision slices: qdd=0:18, force=18:30, torque=30:42")` の観測値を表示して判定根拠を残す。 数式: `print("decision slices: qdd=0:18, force=18:30, torque=30:42")` の演算・変換をPythonで評価する。
print("decision slices: qdd=0:18, force=18:30, torque=30:42")


EoM matrix shape: (18, 42)
decision slices: qdd=0:18, force=18:30, torque=30:42


## Weighted と Hierarchical
`WeightedWbc`はhard constraints + soft taskの加重和をqpOASESで解く単一QP。
`HierarchicalWbc`はnull-space階層を実装するが、`LeggedController::init` は
`WeightedWbc`を生成する。READMEの階層説明を、動いている既定経路と混同しない。

### チューニング
weight変更だけでなく、hard residual、task residual、active constraints、
torque saturation、qp status、solve timeを記録する。実装はQP失敗時fallbackが
明示されていないため、運用では前回安全torqueや停止方針も設計対象になる。


## 章固有の背景
                NMPCの縮約入力だけでは12 motor torqueを直接得られず、全身運動方程式と接触を現在瞬間で満たす必要がある。

                ## 章固有の目的
                42変数、hard constraints、soft tasks、qpOASES入力行列を式とC++ symbolへ一対一対応させる。

                ## この章のASCIIデータフロー
                ```text
                x*,u*,rbd,mode -> WbcBase task matrices
 -> hard: EoM/torque/contact/friction + soft: swing/base/force
 -> WeightedWbc::update -> z=[qdd,F,tau] -> tau
                ```

                ## 上流C++ / faithful pseudocode と数式の行対応
                ```cpp
                // external/legged_control/legged_wbc/src/WbcBase.cpp
a << data.M, -j_.transpose(), -s.transpose(); // [M,-J^T,-S^T]
b = -data.nle;                               // A z = -nle
// external/legged_control/legged_wbc/src/WeightedWbc.cpp
weightedTask = swing*100.0 + baseAccel*1.0 + contactForce*0.01;
H = Asoft.transpose()*Asoft;                 // min 1/2 z^T H z + g^Tz
g = -Asoft.transpose()*bsoft;
qp.init(H,g,Ahard,lb,ub,lbA,ubA,nWsr);       // hard bounds remain constraints
                ```

                **事実のラベル**: `external/legged_control/` の記述はcommit
                `a7f381c0367e98e31c01336e678eef47e304d40d` の上流実装事実。数式展開はそのinterfaceを説明する理論。
                `src/legged_control_mujoco/` に言及した行はproject所有adapterの実装であり、
                ROS1/OCS2 SQP原実装とは同一ではない。

                ## 章固有の結論
                既定は単一WeightedWbc QPであり、GRF追従はweight 0.01のsoft task。解の末尾12 torqueだけがmotorへ進む。
